# Hybrid PCA-30 Spatial — GFLOPs Profiling

This notebook calculates the forward-pass compute for the **existing Hybrid PCA-30 spatial model**. It does **not** retrain the model or change its checkpoint.

It follows the architecture/configuration recorded in `02_salinas_spatial_benchmark_pca_hybrid.ipynb`:
- input tensor: `[1, 30, 15, 15]` (one PCA-30 patch)
- `HybridSpatialSpectralClassifier`
- `feature_dim=128`, `spectral_depth=2`, `spectral_heads=4`
- gated fusion, `dropout=0.1`, `max_bands=256`

The profiler uses **THOP**, matching the project's existing efficiency-notebook convention. Because FLOPs/MACs terminology varies by profiler, the saved JSON also records the tool and convention. This profiles the model forward pass only; it does not include PCA transformation or data loading.

## Before running
Run this notebook from the project root or the `notebooks/` directory. Ensure `outputs/salinas/hybrid_pca30_spatial_best.pt` and the project's `src/dfm/models/hybrid.py` exist. If THOP is missing, run `%pip install thop` in the next cell, then restart the kernel if needed.

In [1]:
# If THOP is not installed, uncomment and run this line:
# %pip install thop

from pathlib import Path
import sys
import json
import platform
import numpy as np
import pandas as pd
import torch

try:
    from thop import profile
except ImportError as exc:
    raise ImportError(
        "THOP is required for this notebook. Run `%pip install thop`, "
        "restart the kernel if necessary, and rerun."
    ) from exc

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "dfm").exists():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    PROJECT_ROOT = next(
        (p for p in candidates if (p / "src" / "dfm").exists()),
        PROJECT_ROOT,
    )
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "salinas"
CHECKPOINT_PATH = PROJECT_ROOT / "outputs" / "salinas" / "hybrid_pca30_spatial_spatial_best.pt"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Checkpoint:", CHECKPOINT_PATH)
print("Checkpoint exists:", CHECKPOINT_PATH.exists())
print("PyTorch:", torch.__version__)
print("THOP:", getattr(__import__("thop"), "__version__", "version not exposed"))


Project root: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup
Checkpoint: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\hybrid_pca30_spatial_spatial_best.pt
Checkpoint exists: True
PyTorch: 2.5.1+cu118
THOP: 0.1.1


In [2]:
from dfm.models.hybrid import HybridSpatialSpectralClassifier

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Expected checkpoint not found: {CHECKPOINT_PATH}\n"
        "Check that the PCA-30 Hybrid spatial training notebook has been run "
        "and that its checkpoint is named `hybrid_pca30_spatial_best.pt`."
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridSpatialSpectralClassifier(
    in_channels=30,
    num_classes=16,
    feature_dim=128,
    spectral_depth=2,
    spectral_heads=4,
    fusion="gated",
    dropout=0.1,
    max_bands=256,
).to(device)

# Load the saved model weights; support the checkpoint format used in the project.
try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
except TypeError:  # compatibility with older PyTorch versions
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

model.load_state_dict(state_dict, strict=True)
model.eval()

parameter_count = sum(p.numel() for p in model.parameters())
trainable_parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
model_size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6

# One sample is used to report compute per sample, independent of inference batch size.
example_input = torch.zeros((1, 30, 15, 15), dtype=torch.float32, device=device)

print("Device:", device)
print("Input shape:", tuple(example_input.shape))
print("Total parameters:", parameter_count)
print("Trainable parameters:", trainable_parameter_count)
print(f"Parameter storage size: {model_size_mb:.6f} MB (decimal)")


Device: cuda
Input shape: (1, 30, 15, 15)
Total parameters: 594576
Trainable parameters: 594576
Parameter storage size: 2.378304 MB (decimal)


In [3]:
# Profile one forward pass. THOP reports its operation count; we store it
# using the same GFLOPs convention as the project's existing efficiency notebooks.
with torch.inference_mode():
    thop_ops, thop_params = profile(model, inputs=(example_input,), verbose=False)

thop_ops = float(thop_ops)
gflops_thop_convention = thop_ops / 1e9

result = {
    "model": "hybrid_pca30_spatial",
    "input_shape": "1x30x15x15",
    "pca_components": 30,
    "patch_size": 15,
    "num_classes": 16,
    "parameters": int(parameter_count),
    "trainable_parameters": int(trainable_parameter_count),
    "model_size_mb": float(model_size_mb),
    "thop_ops": thop_ops,
    "gflops": float(gflops_thop_convention),
    "profiler": "THOP",
    "counting_convention": (
        "THOP-reported operation count divided by 1e9, consistent with the "
        "project's existing efficiency notebooks; interpret MAC/FLOP convention "
        "according to THOP's operator counting."
    ),
    "measurement_scope": "single-sample model forward pass only; excludes PCA preprocessing and data loading",
    "device": str(device),
    "pytorch_version": str(torch.__version__),
    "checkpoint": str(CHECKPOINT_PATH.relative_to(PROJECT_ROOT)),
}

csv_path = OUTPUT_DIR / "hybrid_pca30_spatial_gflops.csv"
json_path = OUTPUT_DIR / "hybrid_pca30_spatial_gflops.json"
pd.DataFrame([result]).to_csv(csv_path, index=False)
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2)

print(f"THOP operation count: {thop_ops:,.0f}")
print(f"GFLOPs (THOP convention): {gflops_thop_convention:.6f}")
print("Saved CSV:", csv_path)
print("Saved JSON:", json_path)
display(pd.DataFrame([result]))


THOP operation count: 33,555,840
GFLOPs (THOP convention): 0.033556
Saved CSV: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\hybrid_pca30_spatial_gflops.csv
Saved JSON: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\hybrid_pca30_spatial_gflops.json


,model,input_shape,pca_components,patch_size,num_classes,parameters,trainable_parameters,model_size_mb,thop_ops,gflops,profiler,counting_convention,measurement_scope,device,pytorch_version,checkpoint
0,hybrid_pca30_spatial,1x30x15x15,30,15,16,594576,594576,2.378304,33555840.0,0.033556,THOP,"THOP-reported operation count divided by 1e9, ...",single-sample model forward pass only; exclude...,cuda,2.5.1+cu118,outputs\salinas\hybrid_pca30_spatial_spatial_b...


## Interpretation note

Use the saved `gflops` value in the comparison table only if the other models' values were calculated with the same profiler/convention and input-size definition. This is a forward-pass estimate for one PCA-30 patch. It does not rerun test evaluation and does not create new accuracy results.